In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [15]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(input_dim, hidden_dim)

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim
        )

    def forward(self, src):

        # src = [src_len, batch_size]

        embedded = self.embedding(src)

        # embedded = [src_len, batch_size, hidden_dim]

        outputs, hidden = self.gru(embedded)

        # hidden = [1, batch_size, hidden_dim]

        return hidden

In [16]:
# Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim):
        super().__init__()

        self.output_dim = output_dim

        self.embedding = nn.Embedding(output_dim, hidden_dim)

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim
        )

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden):

        # input_token = [batch_size]

        input_token = input_token.unsqueeze(0)

        # [1, batch_size]

        embedded = self.embedding(input_token)

        # [1, batch_size, hidden_dim]

        output, hidden = self.gru(embedded, hidden)

        prediction = self.fc(output.squeeze(0))

        # prediction = [batch_size, output_dim]

        return prediction, hidden

In [17]:
# Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size
        ).to(trg.device)

        hidden = self.encoder(src)

        # First decoder input = <SOS>
        input_token = trg[0]

        for t in range(1, trg_len):

            output, hidden = self.decoder(
                input_token,
                hidden
            )

            outputs[t] = output

            teacher_force = random.random() < teacher_forcing_ratio

            top1 = output.argmax(1)

            input_token = trg[t] if teacher_force else top1

        return outputs

In [18]:
# Hyperparameters
INPUT_DIM = 20
OUTPUT_DIM = 20
HIDDEN_DIM = 64

encoder = Encoder(INPUT_DIM, HIDDEN_DIM)
decoder = Decoder(OUTPUT_DIM, HIDDEN_DIM)

model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [19]:
# Dummy Data
src = torch.tensor([
    [1, 1],
    [5, 7],
    [8, 9],
    [2, 2]
])

trg = torch.tensor([
    [1, 1],
    [5, 7],
    [8, 9],
    [2, 2]
])

In [20]:

# Training
EPOCHS = 100

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    output = model(src, trg)

    # Ignore first token
    output = output[1:].reshape(-1, OUTPUT_DIM)

    target = trg[1:].reshape(-1)

    loss = criterion(output, target)

    loss.backward()

    optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | Loss = {loss.item():.4f}"
        )

Epoch   0 | Loss = 3.0763
Epoch  10 | Loss = 2.3235
Epoch  20 | Loss = 1.3896
Epoch  30 | Loss = 0.7536
Epoch  40 | Loss = 0.3778
Epoch  50 | Loss = 0.1973
Epoch  60 | Loss = 0.1163
Epoch  70 | Loss = 0.0781
Epoch  80 | Loss = 0.0578
Epoch  90 | Loss = 0.0457


In [21]:
# Inference
model.eval()

with torch.no_grad():

    output = model(
        src,
        trg,
        teacher_forcing_ratio=0
    )

    predictions = output.argmax(2)

    print("\nPredictions:")
    print(predictions)


Predictions:
tensor([[0, 0],
        [5, 7],
        [8, 9],
        [2, 2]])
